In [ ]:
# data analysis
import pandas as pd
import numpy as np
import os
import time
import datetime
import psutil
from cpuinfo import get_cpu_info
import platform
import subprocess
from math import pi
from functools import partial
from sklearn.feature_selection import mutual_info_classif
from scipy.stats import chi2_contingency
from itertools import combinations
from scipy.stats import skew
from sklearn.preprocessing import PowerTransformer
from imblearn.over_sampling import SMOTENC
from sklearn.preprocessing import MinMaxScaler
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster
from pycaret.classification import *
from sklearn.model_selection import learning_curve
from sklearn.metrics import roc_curve, auc
# data visualization
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.table import Table
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_auc_score, ConfusionMatrixDisplay, auc, roc_curve
from sklearn.metrics import cohen_kappa_score, f1_score, precision_score, recall_score, accuracy_score,matthews_corrcoef
from sklearn.model_selection import learning_curve
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBRFClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier
#warnings
import warnings, traceback
warnings.filterwarnings("ignore")
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
file_path = "/content/drive/MyDrive/project_first/clean_sql_export_for_datamining_day_to_quartile.xlsx"
dataset = pd.read_excel(file_path,sheet_name="data")
pd.set_option("display.max_columns",None) # for showing all columns
pd.set_option("display.expand_frame_repr",False) # to avoid data clutter
dataset

,serial,zsituation,brithkind,sex_b,weight_b,zcomment,joutcomment,cout_joutcomment,drylendays,prelendays,DFS_milkperiod,DFS,dis_ab,count_ab,tmom,father,age_breeding,day_tdate,mount_tdate,year_tdate,MilkPeriod,tkind,SpermNum,PersonCode,dim_tdate,avgmilk,faz_mah,time_faz,not_return
0,1001369063103130,2,1,f,41,2,1,9,54,273,94,449,.,0,1001366110402530,1775836,163.409836,forth,3,1425,10,2,120173998,1011,339,19.6,21,2,0
1,1001369063103130,2,1,f,41,2,1,9,54,273,94,449,.,0,1001366110402530,1775836,164.131147,tirth,4,1425,10,2,124717307,1011,361,17.2,2,2,0
2,1001369071303650,1,1,m,.,2,1,8,108,265,47,484,.,0,1001366032200610,1775836,166.754098,tirth,7,1425,11,2,17256591,1011,47,27.4,21,2,0
3,1001369071303650,1,1,m,.,2,1,8,108,265,47,484,.,0,1001366032200610,1775836,168.000000,first,9,1425,11,2,126365823,1010,85,23.5,1,2,0
4,1001369071303650,1,1,m,.,2,1,8,108,265,47,484,.,0,1001366032200610,1775836,168.557377,tirth,9,1425,11,2,9043859,1011,102,20.2,2,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118991,1761386102005300,2,1,f,44,2,0,0,95,279,71,445,.,0,1761381072004850,176138405040444,85.836065,tirth,5,1436,5,2,62916235,1028,248,23.25,21,4,0
118992,1761386102005300,2,1,f,44,2,0,0,95,279,71,445,.,0,1761381072004850,176138405040444,87.409836,second,7,1436,5,2,69791579,1028,296,25.1,11,2,0
118993,1761386102205370,2,1,m,48,1,0,0,0,279,64,437,.,0,1761383110206710,128600717,28.163934,first,6,1431,1,2,129008732,1028,64,34.8,11,4,1
118994,1761386102205370,2,1,f,40,1,0,0,58,280,49,437,.,0,1761383110206710,128600717,38.950819,second,5,1432,2,2,66011448,1028,49,59.9,11,2,1


In [ ]:
dataset_copy = dataset.copy() #a copy of dataset has made to make sure that main dataset doesnot change

In [ ]:
dataset_copy.info()#complete information of dataset's columns has printed that contains column name, number of data, and data type

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118996 entries, 0 to 118995
Data columns (total 29 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   serial            118996 non-null  int64  
 1   zsituation        118996 non-null  int64  
 2   brithkind         118996 non-null  int64  
 3   sex_b             118996 non-null  object 
 4   weight_b          118996 non-null  object 
 5   zcomment          118996 non-null  object 
 6   joutcomment       118996 non-null  object 
 7   cout_joutcomment  118996 non-null  int64  
 8   drylendays        118996 non-null  int64  
 9   prelendays        118996 non-null  int64  
 10  DFS_milkperiod    118996 non-null  int64  
 11  DFS               118996 non-null  int64  
 12  dis_ab            118996 non-null  object 
 13  count_ab          118996 non-null  int64  
 14  tmom              118996 non-null  object 
 15  father            118996 non-null  object 
 16  age_breeding      11

In [ ]:
dataset_copy.rename(columns={'serial':'Animal', 'zsituation':'PAS', 'brithkind': 'BT',
                             'sex_b':'SEX', 'weight_b':'WB', 'zcomment':'DS', 'joutcomment':'RP',
                             'cout_joutcomment':'TNR', 'drylendays':'DR', 'prelendays': 'GL',
                             'DFS_milkperiod':'DMP', 'DFS':'DFS', 'dis_ab': 'IFI',
                             'count_ab':'NPA', 'tmom':'DIM', 'father':'SIR', 'age_breeding':'AI',
                             'day_tdate':'DI', 'mount_tdate':'MI', 'year_tdate':'YI',
                             'MilkPeriod':'LP', 'tkind':'TP', 'SpermNum':'S', 'PersonCode':'AIT',
                             'dim_tdate':'DM', 'avgmilk':'AM', 'faz_mah':'CL', 'time_faz':'CI',
                             'not_return':'PS'}, inplace=True)#columns names has changed to simple names so they can be used easily in other stages

In [ ]:
dataset_copy

,Animal,PAS,BT,SEX,WB,DS,RP,TNR,DR,GL,DMP,DFS,IFI,NPA,DIM,SIR,AI,DI,MI,YI,LP,TP,S,AIT,DM,AM,CL,CI,PS
0,1001369063103130,2,1,f,41,2,1,9,54,273,94,449,.,0,1001366110402530,1775836,163.409836,forth,3,1425,10,2,120173998,1011,339,19.6,21,2,0
1,1001369063103130,2,1,f,41,2,1,9,54,273,94,449,.,0,1001366110402530,1775836,164.131147,tirth,4,1425,10,2,124717307,1011,361,17.2,2,2,0
2,1001369071303650,1,1,m,.,2,1,8,108,265,47,484,.,0,1001366032200610,1775836,166.754098,tirth,7,1425,11,2,17256591,1011,47,27.4,21,2,0
3,1001369071303650,1,1,m,.,2,1,8,108,265,47,484,.,0,1001366032200610,1775836,168.000000,first,9,1425,11,2,126365823,1010,85,23.5,1,2,0
4,1001369071303650,1,1,m,.,2,1,8,108,265,47,484,.,0,1001366032200610,1775836,168.557377,tirth,9,1425,11,2,9043859,1011,102,20.2,2,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118991,1761386102005300,2,1,f,44,2,0,0,95,279,71,445,.,0,1761381072004850,176138405040444,85.836065,tirth,5,1436,5,2,62916235,1028,248,23.25,21,4,0
118992,1761386102005300,2,1,f,44,2,0,0,95,279,71,445,.,0,1761381072004850,176138405040444,87.409836,second,7,1436,5,2,69791579,1028,296,25.1,11,2,0
118993,1761386102205370,2,1,m,48,1,0,0,0,279,64,437,.,0,1761383110206710,128600717,28.163934,first,6,1431,1,2,129008732,1028,64,34.8,11,4,1
118994,1761386102205370,2,1,f,40,1,0,0,58,280,49,437,.,0,1761383110206710,128600717,38.950819,second,5,1432,2,2,66011448,1028,49,59.9,11,2,1


In [ ]:
for column in dataset_copy.columns:#data of contiuous columns and categories of categorical columns have printed
    print(f"{column}")
    print(dataset_copy[column].unique())
    print("\n")

Animal
[1001369063103130 1001369071303650 1001369102208260 ... 1761386101805250
 1761386102005300 1761386102205370]


PAS
[ 2  1 12]


BT
[1 2 3]


SEX
['f' 'm' '2m' '2f' '2' '9']


WB
['41' '.' '40' '45' '46' '38' '44' '43' '29' '35' '48' '47' '50' '49' '52'
 '60' '51' '42' '39' '37' '53' '55' '77' '63' '74' '62' '58' '34' '33'
 '81' '36' '57' '30' '66' '61' '67' '76' '56' '28' '64' '4' '26' '32' '18'
 '78' '31' '23' '88' '73' '54' '59' '27' '13' '3.6' '21' '84' '783' '22'
 '70' '12' '69' '75' '82' '38.8' '72' '80' '79' '24' '4.4' '68' '71' '90'
 '65' '85' '20' '91' '25' '11' '0' '16' '38.1' '273' '249' '307' '234'
 '87' '3.7' '19' '83' '3.2' '286' '2.7' '15' '93' '2.9' '413' '86' '10'
 '92' '13.4' '38.9' '17' '38.2' '39.5' '40.2' '2' '630' '38.6' '637' '121'
 '145' '138' '115' '111' '74.3' '120' '7' '3.1' '44.5' '122' '1.47' '160'
 '135' '145.5' '119' '142' '365' '3.4' '98.5' '118' '1.42' '47.75' '10.2'
 '38.7' '14' '662' '160.4' '454' '139' '39.2' '358' '174' '38.3' '131.5'
 '39.1' 

In [ ]:
dataset_copy["SEX"].value_counts()#check the number of samples in each category of the SEX column and name of categories

,count
SEX,
m,60110
f,54947
2,1936
2f,756
2m,724
9,523


In [ ]:
dataset_copy["DI"].value_counts()#check the number of samples in each category of the DI column and name of categories

,count
DI,
tirth,32495
forth,30478
first,28058
second,27965


In [ ]:
def categorical_to_numerical(df):
    """
    This function maps non_numeric categories in
    the SEX and DI columns to thiers corresponding
    numeric values
    df:dataset
    """

    column1 = "SEX"
    column2 = "DI"
    label_map1 = {"m":0,"f":1,"2":2,"2m":3,"2f":4}
    label_map2 = {"first":0,"second":1,"tirth":2,"forth":3}
    df[column1] = df[column1].map(label_map1)
    df[column2] = df[column2].map(label_map2)

    return df


dataset_copy = categorical_to_numerical(dataset_copy)

In [ ]:
columns_to_convert = ['SEX', 'WB', 'DS', 'RP', 'IFI', 'DIM', 'SIR',
                      'DI', 'S', 'AIT', 'AM']

for col in columns_to_convert: #convert other non_numeric columns   data to numeric data
  dataset_copy[col] = pd.to_numeric(dataset_copy[col], errors='coerce')

In [ ]:
dataset_copy.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 118996 entries, 0 to 118995
Data columns (total 29 columns):
 #   Column  Non-Null Count   Dtype  
---  ------  --------------   -----  
 0   Animal  118996 non-null  int64  
 1   PAS     118996 non-null  int64  
 2   BT      118996 non-null  int64  
 3   SEX     118473 non-null  float64
 4   WB      117069 non-null  float64
 5   DS      118986 non-null  float64
 6   RP      117087 non-null  float64
 7   TNR     118996 non-null  int64  
 8   DR      118996 non-null  int64  
 9   GL      118996 non-null  int64  
 10  DMP     118996 non-null  int64  
 11  DFS     118996 non-null  int64  
 12  IFI     3881 non-null    float64
 13  NPA     118996 non-null  int64  
 14  DIM     115445 non-null  float64
 15  SIR     115005 non-null  float64
 16  AI      118996 non-null  float64
 17  DI      118996 non-null  int64  
 18  MI      118996 non-null  int64  
 19  YI      118996 non-null  int64  
 20  LP      118996 non-null  int64  
 21  TP      11

In [ ]:
dataset_copy.isnull().sum()#counts number of null data for each column

,0
Animal,0
PAS,0
BT,0
SEX,523
WB,1927
DS,10
RP,1909
TNR,0
DR,0
GL,0


In [ ]:
columns_to_check = ['WB', 'RP', 'AM', 'SEX', 'DS', "IFI"]
null_percentages = {}
for col in columns_to_check: # prints the percentage of null data for columns in columns_to_check
  null_count = dataset_copy[col].isnull().sum()
  total_count = len(dataset_copy[col])
  percent = (null_count / total_count) * 100
  null_percentages[col] = percent


for col, percent in null_percentages.items():
  print(f" the col {col} has {percent:2f}% null data")

 the col WB has 1.619382% null data
 the col RP has 1.604256% null data
 the col AM has 0.191603% null data
 the col SEX has 0.439511% null data
 the col DS has 0.008404% null data
 the col IFI has 96.738546% null data


In [ ]:
dataset_copy = dataset_copy.drop(columns=["Animal", "IFI", "DIM", "SIR", "AIT", "S"])#removes columns that contains coded or identifier data

In [ ]:
dataset_copy = dataset_copy.dropna(subset=['SEX', 'DS', 'WB', 'RP', 'AM'])#removes null values from the specified columns

In [ ]:
dataset_copy.isnull().sum()# counts the number of null values for each column

,0
PAS,0
BT,0
SEX,0
WB,0
DS,0
RP,0
TNR,0
DR,0
GL,0
DMP,0


In [ ]:
dataset_copy["PS"].value_counts()#counts the number of values for each category in target column

,count
PS,
0,80510
1,34508
2,1


In [ ]:
index_to_drop = dataset_copy.index[dataset_copy["PS"] == 2] # removes incorrect category from target column
dataset_copy.drop(index=index_to_drop, inplace=True)

In [ ]:
dataset_copy.duplicated().any() #examines the dataset if there is any duplicated data

np.True_

In [ ]:
dataset_copy = dataset_copy.drop_duplicates()#removes duplicated data from dataset

In [ ]:
dataset_copy.describe()#provides summary statistics of the dataset's features

,PAS,BT,SEX,WB,DS,RP,TNR,DR,GL,DMP,DFS,NPA,AI,DI,MI,YI,LP,TP,DM,AM,CL,CI,PS
count,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000,115004.000000
mean,1.960523,1.024686,0.541407,43.775749,1.236087,0.160299,0.599257,43.876796,276.882213,64.570102,441.750287,0.124448,50.124532,1.549303,6.462888,1432.153464,2.534921,2.007765,146.891986,40.578417,8.356335,2.816406,0.299972
std,0.194727,0.155671,0.624741,13.478930,0.476293,0.366885,1.144765,42.455473,7.743487,27.449279,35.381076,0.383536,23.260985,1.109711,3.561883,3.819209,1.666964,0.088172,98.798850,10.768800,8.019247,0.937346,0.458247
min,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,29.000000,-43.000000,76.000000,0.000000,16.688524,0.000000,1.000000,1425.000000,1.000000,1.000000,13.000000,0.100000,1.000000,1.000000,0.000000
25%,2.000000,1.000000,0.000000,40.000000,1.000000,0.000000,0.000000,0.000000,274.000000,47.000000,421.000000,0.000000,31.311475,1.000000,3.000000,1429.000000,1.000000,2.000000,71.000000,33.800000,1.000000,2.000000,0.000000
50%,2.000000,1.000000,0.000000,43.000000,1.000000,0.000000,0.000000,50.000000,277.000000,57.000000,436.000000,0.000000,43.770491,2.000000,6.000000,1432.000000,2.000000,2.000000,118.000000,40.800000,2.000000,2.000000,0.000000
75%,2.000000,1.000000,1.000000,47.000000,1.000000,0.000000,1.000000,61.000000,281.000000,72.000000,461.000000,0.000000,62.065573,3.000000,10.000000,1436.000000,3.000000,2.000000,196.000000,47.800000,11.000000,4.000000,1.000000
max,2.000000,3.000000,4.000000,783.000000,4.000000,1.000000,10.000000,632.000000,333.000000,671.000000,1153.000000,5.000000,183.344262,3.000000,12.000000,1438.000000,14.000000,3.000000,814.000000,130.100000,21.000000,4.000000,1.000000


In [ ]:
negative_values = dataset_copy[dataset_copy['DMP'] < 0] # finds the number of negative values in the clumn DMP
num_negative = negative_values.shape[0]
print(f"number of nagative values in column DMP is {num_negative}")

number of nagative values in column DMP is 10


In [ ]:
negative_index = dataset_copy.index[dataset_copy["DMP"] < 0] #removs the nagative values from the column DMP
dataset_copy.drop(index=negative_index, inplace=True)

In [ ]:
for column in dataset_copy.columns:
    print(f"{column}")
    print(dataset_copy[column].unique())
    print("\n")

PAS
[2 1]


BT
[1 2 3]


SEX
[1. 0. 3. 4. 2.]


WB
[ 41.    40.    45.    46.    44.    43.    29.    38.    35.    48.
  47.    50.    49.    52.    60.    51.    42.    39.    37.    53.
  55.    77.    63.    74.    62.    58.    34.    33.    81.    36.
  57.    30.    66.    61.    67.    76.    56.    28.     4.    26.
  32.    18.    78.    31.    23.    88.    73.    54.    59.    27.
  13.     3.6   64.    21.    84.   783.    22.    70.    12.    69.
  75.    82.    38.8   72.    80.    79.    24.     4.4   68.    71.
  90.    65.    85.    20.    91.    25.    11.     0.    38.1  273.
 249.   307.   234.    87.     3.7   19.    83.     3.2  286.     2.7
  15.    93.     2.9  413.    86.    92.    13.4   38.9   17.    38.2
  39.5   40.2    2.   630.    38.6  637.   121.   145.   138.   115.
 111.    74.3  120.     7.     3.1   44.5  122.     1.47 160.    10.
 135.   145.5  142.   365.     3.4   98.5  118.     1.42  47.75  10.2
  38.7   14.   662.   160.4  454.   139.    39.2 

In [ ]:
#groups the dataset by the BT column and lists the unique values of the SEX column within each group
dataset_copy.groupby("BT")["SEX"].unique()

,SEX
BT,
1,"[1.0, 0.0, 2.0, 3.0]"
2,"[3.0, 4.0, 2.0]"
3,"[1.0, 2.0]"


In [ ]:
dataset_copy[dataset_copy["BT"]==3] #displays the rows of the dataset where the BT column has a value of 3

,PAS,BT,SEX,WB,DS,RP,TNR,DR,GL,DMP,DFS,NPA,AI,DI,MI,YI,LP,TP,DM,AM,CL,CI,PS
52265,2,3,1.0,24.0,1.0,1.0,2,97,256,54,488,0,59.147540,0,10,1432,3,2,54,51.80,11,2,0
52266,2,3,1.0,24.0,1.0,1.0,2,97,256,54,488,0,60.655737,2,11,1432,3,2,100,45.20,21,2,0
52267,2,3,1.0,24.0,1.0,1.0,2,97,256,54,488,0,61.573770,2,12,1432,3,2,128,37.40,21,4,1
52268,2,3,1.0,24.0,1.0,1.0,2,97,256,54,488,1,63.409836,2,2,1433,3,2,184,42.20,2,4,1
100525,2,3,2.0,36.0,1.0,1.0,1,46,270,51,437,0,36.852459,2,11,1436,2,2,51,49.40,21,3,0
100526,2,3,2.0,36.0,1.0,1.0,1,46,270,51,437,0,40.360655,1,3,1437,2,2,158,52.05,11,3,0
100527,2,3,2.0,36.0,1.0,1.0,1,46,270,51,437,0,42.754098,3,5,1437,2,2,231,27.90,21,4,0
100528,2,3,2.0,36.0,1.0,1.0,1,46,270,51,437,0,44.426229,2,7,1437,2,2,282,44.80,2,2,0
100529,2,3,2.0,36.0,1.0,1.0,1,46,270,51,437,0,45.868852,3,8,1437,2,2,326,35.30,1,2,0


In [ ]:
#displays the rows of the dataset where the BT column has a value of 1 and the Sex column has a value of 2
dataset_copy[(dataset_copy["BT"]==1) & (dataset_copy["SEX"]==2)]

,PAS,BT,SEX,WB,DS,RP,TNR,DR,GL,DMP,DFS,NPA,AI,DI,MI,YI,LP,TP,DM,AM,CL,CI,PS
5235,2,1,2.0,41.0,1.0,0.0,4,59,277,61,489,0,85.475409,2,10,1427,6,2,61,43.7,2,2,1
5957,2,1,2.0,40.0,1.0,0.0,4,47,270,46,467,0,105.573770,2,11,1429,7,2,46,49.9,2,2,0
5958,2,1,2.0,40.0,1.0,0.0,4,47,270,46,467,0,106.295081,0,12,1429,7,2,68,57.8,11,2,0
5959,2,1,2.0,40.0,1.0,0.0,4,47,270,46,467,0,109.737704,3,3,1430,7,2,173,55.2,21,2,1
12213,2,1,2.0,42.0,1.0,0.0,3,61,284,40,456,0,98.196721,1,12,1430,7,2,40,18.8,11,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118319,2,1,2.0,34.0,1.0,0.0,0,52,274,54,473,0,63.704918,2,6,1433,4,2,165,54.0,2,2,1
118355,2,1,2.0,34.0,1.0,0.0,1,110,273,56,467,2,58.393442,2,1,1433,3,2,56,50.2,21,2,0
118356,2,1,2.0,34.0,1.0,0.0,1,110,273,56,467,2,61.245901,2,4,1433,3,2,143,29.2,21,3,0
118357,2,1,2.0,34.0,1.0,0.0,1,110,273,56,467,2,62.655737,0,6,1433,3,2,186,42.6,11,4,0


In [ ]:
COUNT = dataset_copy[(dataset_copy["BT"]==1) & (dataset_copy["SEX"]==2)].shape[0]
print(COUNT)#counts the rows of the dataset where the BT column has a value of 1 and the Sex column has a value of 2

398


In [ ]:
COUNT1 = dataset_copy[(dataset_copy["BT"]==1) & (dataset_copy["SEX"]==3)].shape[0]
print(COUNT1)#counts the rows of the dataset where the BT column has a value of 1 and the Sex column has a value of 3

2


In [ ]:
dataset_copy =dataset_copy[dataset_copy["BT"] != 3] #removes the rows of the dataset where the BT column has a value pf 3

In [ ]:
#removes the rows of the dataset where the BT column has a value of 1 and the Sex column has a value of 3
dataset_copy = dataset_copy[~((dataset_copy["BT"]==1) & (dataset_copy["SEX"]==3))]

In [ ]:
#removes the rows of the dataset where the BT column has a value of 1 and the Sex column has a value of 2
dataset_copy = dataset_copy[~((dataset_copy["BT"]==1) & (dataset_copy["SEX"]==2))]

In [ ]:
dataset_copy

,PAS,BT,SEX,WB,DS,RP,TNR,DR,GL,DMP,DFS,NPA,AI,DI,MI,YI,LP,TP,DM,AM,CL,CI,PS
0,2,1,1.0,41.0,2.0,1.0,9,54,273,94,449,0,163.409836,3,3,1425,10,2,339,19.60,21,2,0
1,2,1,1.0,41.0,2.0,1.0,9,54,273,94,449,0,164.131147,2,4,1425,10,2,361,17.20,2,2,0
6,2,1,0.0,40.0,2.0,1.0,8,111,281,74,473,1,159.573770,2,3,1425,10,2,74,30.40,21,2,0
7,2,1,0.0,40.0,2.0,0.0,7,85,282,79,473,0,157.475409,0,9,1425,11,2,79,36.10,1,2,1
8,2,1,0.0,41.0,2.0,1.0,10,57,275,46,485,0,147.442622,0,2,1425,10,2,178,30.60,1,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118990,2,1,1.0,44.0,2.0,0.0,0,95,279,71,445,0,85.049180,3,4,1436,5,2,224,24.80,1,2,0
118991,2,1,1.0,44.0,2.0,0.0,0,95,279,71,445,0,85.836065,2,5,1436,5,2,248,23.25,21,4,0
118992,2,1,1.0,44.0,2.0,0.0,0,95,279,71,445,0,87.409836,1,7,1436,5,2,296,25.10,11,2,0
118993,2,1,0.0,48.0,1.0,0.0,0,0,279,64,437,0,28.163934,0,6,1431,1,2,64,34.80,11,4,1


In [ ]:
dataset_copy.to_excel("/content/drive/MyDrive/project_first/cleared_data.xlsx", index=False) # saves cleared data

In [ ]:
file_path = "/content/drive/MyDrive/project_first/cleared_data.xlsx"
dataset_cleared = pd.read_excel(file_path) # displays cleared data
pd.set_option("display.max_columns",None) # for showing all columns
pd.set_option("display.expand_frame_repr",False) # to avoid data clutter
dataset_cleared

,PAS,BT,SEX,WB,DS,RP,TNR,DR,GL,DMP,DFS,NPA,AI,DI,MI,YI,LP,TP,DM,AM,CL,CI,PS
0,2,1,1,41.0,2,1,9,54,273,94,449,0,163.409836,3,3,1425,10,2,339,19.60,21,2,0
1,2,1,1,41.0,2,1,9,54,273,94,449,0,164.131147,2,4,1425,10,2,361,17.20,2,2,0
2,2,1,0,40.0,2,1,8,111,281,74,473,1,159.573770,2,3,1425,10,2,74,30.40,21,2,0
3,2,1,0,40.0,2,0,7,85,282,79,473,0,157.475409,0,9,1425,11,2,79,36.10,1,2,1
4,2,1,0,41.0,2,1,10,57,275,46,485,0,147.442622,0,2,1425,10,2,178,30.60,1,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114580,2,1,1,44.0,2,0,0,95,279,71,445,0,85.049180,3,4,1436,5,2,224,24.80,1,2,0
114581,2,1,1,44.0,2,0,0,95,279,71,445,0,85.836065,2,5,1436,5,2,248,23.25,21,4,0
114582,2,1,1,44.0,2,0,0,95,279,71,445,0,87.409836,1,7,1436,5,2,296,25.10,11,2,0
114583,2,1,0,48.0,1,0,0,0,279,64,437,0,28.163934,0,6,1431,1,2,64,34.80,11,4,1


In [ ]:
dataset_cleared_copy = dataset_cleared.copy()  # makes a copy from dataset

In [ ]:
continuous_columns = ["WB", "DR", "GL", "DMP", "DFS", "AI", "DM", "AM"] #contains the continuous columns
categorial_columns = ["PAS", "BT", "SEX", "DS", "RP", "TNR", "NPA", "DI", "MI",
                      "YI", "LP", "TP", "CL", "CI", "PS"]  # contains categorical column

In [ ]:
def plot_distributions(df, continuous_cols, categorical_cols,
                       save_path):
    """
    This function plots the distribution diagrams
    (hisplot for continuous columns and count plot for categorical columns)
    for all columns of dataset
    """
  n_cols = 4
  n_rows = 2
  fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3))
  axes = axes.flatten()
  colors_cont = sns.color_palette("tab20", len(continuous_cols))
  for i, col in enumerate(continuous_cols):
    sns.histplot(df[col], kde=True, bins=30, color=colors_cont[i], ax=axes[i])
    axes[i].set_title(col, fontsize=12, fontweight="bold")
    axes[i].set_xlabel('')
    axes[i].set_ylabel("Count")
    axes[i].grid(axis='y', linestyle='--', alpha=0.7)

  for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
  plt.tight_layout()
  if save_path:
    os.makedirs(save_path, exist_ok=True)
    plt.savefig(os.path.join(save_path, "continuous_cols_distributions.png"), dpi=900)
  plt.close()
  cat_first8 = categorical_cols[:8]
  n_cols = 4
  n_rows = 2
  fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3))
  axes = axes.flatten()
  colors_cat = sns.color_palette("Set3", len(cat_first8))
  for i, col in enumerate(cat_first8):
    order = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order, palette=[colors_cat[i]]*len(order), ax=axes[i])
    axes[i].set_title(col, fontsize=12, fontweight="bold")
    axes[i].set_xlabel('')
    axes[i].set_ylabel("Frequency")
    axes[i].tick_params(axis="x", rotation=45)
    axes[i].grid(axis='y', linestyle='--', alpha=0.7)

  for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
  plt.tight_layout()
  if save_path:
    plt.savefig(os.path.join(save_path, "categorical_fisrt8.png"), dpi=900)
  plt.close()
  cat_remaining = categorical_cols[8:]
  n_cols = 4
  n_rows = 2
  fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3))
  axes = axes.flatten()
  colors_cat_1 = sns.color_palette("Set3", len(cat_remaining))
  for i, col in enumerate(cat_remaining):
    order_2 = df[col].value_counts().index
    sns.countplot(data=df, x=col, order=order_2, palette=[colors_cat_1[i]]*len(order_2), ax=axes[i])
    axes[i].set_title(col, fontsize=12, fontweight="bold")
    axes[i].set_xlabel('')
    axes[i].set_ylabel("Frequency")
    axes[i].tick_params(axis="x", rotation=45)
    axes[i].grid(axis='y', linestyle='--', alpha=0.7)
  for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])
  plt.tight_layout()
  if save_path:
    plt.savefig(os.path.join(save_path, "categorical_remaining7.png"), dpi=900)
  plt.close()



save_path1 = "/content/drive/MyDrive/project_first"
plot_distributions(dataset_cleared_copy, continuous_columns, categorial_columns, save_path1)

In [ ]:
def plot_skewness_table(df, continuous_cols, save_path=None):
    """
    This function first calculates the skewness for continuous columns
    then makes a table for them
    """
  skew_values = [skew(df[col], bias=False) for col in continuous_cols]
  skew_table = pd.DataFrame({"Feature": continuous_cols, "Skewness": skew_values})
  skew_table["Abs_skew"] = skew_table["Skewness"].abs()
  skew_table.sort_values(by="Abs_skew", ascending=False, inplace=True)
  skew_table.reset_index(drop=True, inplace=True)
  table_matrix = skew_table[["Feature", "Skewness"]].copy()
  table_matrix.set_index("Feature", inplace=True)
  plt.figure(figsize=(max(8, len(continuous_cols)*0.6), 2))
  cmap = sns.diverging_palette(220, 20, as_cmap=True)
  sns.heatmap(table_matrix.T, annot=True, fmt=".3f", cmap=cmap, cbar=True,
              linewidths=0.5, linecolor="gray", center=0,
              annot_kws={"fontsize":11, "fontweight":"bold", "color":"black"})
  plt.title("Skewness of Continuous Features", fontsize=14, fontweight="bold")
  plt.yticks(rotation=0, fontsize=10)
  plt.xticks(rotation=45, ha="right", fontsize=10)
  plt.tight_layout()
  if save_path:
    os.makedirs(save_path, exist_ok=True)
  plt.savefig(os.path.join(save_path, "continuous_skewness_table.png"), dpi=900, bbox_inches="tight")
  plt.close()
  return skew_table




plot_skewness_table(dataset_cleared_copy, continuous_columns, save_path1)


,Feature,Skewness,Abs_skew
0,WB,24.619073,24.619073
1,GL,-5.041256,5.041256
2,DMP,2.775063,2.775063
3,DFS,2.705960,2.705960
4,DR,1.623063,1.623063
5,DM,1.414756,1.414756
6,AI,1.290745,1.290745
7,AM,-0.130945,0.130945


In [ ]:
continuous_columns = ["WB", "DR", "GL", "DMP", "DFS", "AI", "DM", "AM"]  # counts the number of zero values for specified columns
for col in continuous_columns:
  n_zeros = (dataset_cleared_copy[col] == 0).sum()
  if n_zeros> 0:
    print(f"column {col} has {n_zeros} zero data")
  else:
    print(f"column {col} has no zero data")

column WB has 14 zero data
column DR has 38321 zero data
column GL has no zero data
column DMP has no zero data
column DFS has no zero data
column AI has no zero data
column DM has no zero data
column AM has no zero data


In [ ]:
def plot_new_skewness_table(df, continuous_cols, save_path=None):
    """
    This function first calculates the skewness after addressing the issue
    than plots  the updated skewness table
    """
  skew_values = [skew(df[col], bias=False) for col in continuous_cols]
  skew_table = pd.DataFrame({"Feature": continuous_cols, "Skewness": skew_values})
  skew_table["Abs_skew"] = skew_table["Skewness"].abs()
  skew_table.sort_values(by="Abs_skew", ascending=False, inplace=True)
  skew_table.reset_index(drop=True, inplace=True)
  table_matrix = skew_table[["Feature", "Skewness"]].copy()
  table_matrix.set_index("Feature", inplace=True)
  plt.figure(figsize=(max(8, len(continuous_cols)*0.6), 2))
  cmap = sns.diverging_palette(220, 20, as_cmap=True)
  sns.heatmap(table_matrix.T, annot=True, fmt=".3f", cmap=cmap, cbar=True,
              linewidths=0.5, linecolor="gray", center=0,
              annot_kws={"fontsize":11, "fontweight":"bold", "color":"black"})
  plt.title("Skewness of Continuous Features", fontsize=14)
  plt.yticks(rotation=0)
  plt.xticks(rotation=45, ha="right")
  plt.tight_layout()
  if save_path:
    os.makedirs(save_path, exist_ok=True)
  plt.savefig(os.path.join(save_path, "continuous_skewness_table_after.png"), dpi=900, bbox_inches="tight")
  plt.close()
  return skew_table



In [ ]:
# AM columns is approximately normal
# columns WB and DR have zero data
skew_columns = ["WB", "DMP","GL", "DFS", "DR", "DM", "AI"]
pt = PowerTransformer(method="yeo-johnson", standardize=False)
lambda_yeo_dict = {}
for col in skew_columns: #adresses the issue of skewness in continuous columns by using yeo-johnson method
  data = dataset_cleared_copy[[col]].values
  dataset_cleared_copy[col] = pt.fit_transform(data)
  lambda_yeo_dict[col] = float(pt.lambdas_[0])

skew_after = [skew(dataset_cleared_copy[col].values, bias=False) for col in skew_columns]
report = pd.DataFrame({
    "Column": skew_columns,
    "Transformation": ["Yeo-Johnson"] * len(skew_columns),
    "Lambda":[lambda_yeo_dict[col] for col in skew_columns],
    "skewness_after":skew_after
    })
print(report)
report.to_csv("/content/drive/MyDrive/project_first/skewness_transformation_report.csv", index=False)



  Column Transformation    Lambda  skewness_after
0     WB    Yeo-Johnson  0.125694        0.355934
1    DMP    Yeo-Johnson -0.849891        0.020304
2     GL    Yeo-Johnson  9.691543        0.389441
3    DFS    Yeo-Johnson -0.000820       -0.000636
4     DR    Yeo-Johnson  0.323128       -0.350530
5     DM    Yeo-Johnson -0.105640        0.019432
6     AI    Yeo-Johnson -0.471360        0.063438


In [ ]:
#plots the distribution diagrams for continuous columns after adressing the issue
n_cols = 4
n_rows = 2
continuous_columns_1 = ["WB", "DR", "GL", "DMP", "DFS", "AI", "DM", "AM"]
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3))
axes = axes.flatten()
colors_cont_2 = sns.color_palette("tab20", len(continuous_columns_1))
for i, col in enumerate(continuous_columns_1):
  sns.histplot(dataset_cleared_copy[col], kde=True, bins=30, color=colors_cont_2[i], ax=axes[i])
  axes[i].set_title(col, fontsize=12, fontweight="bold")
  axes[i].set_xlabel('')
  axes[i].set_ylabel("Count")
  axes[i].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/project_first/continuous_cols_distributions_after.png", dpi=900)
plt.close()

In [ ]:
new_path = "/content/drive/MyDrive/project_first"
plot_new_skewness_table(dataset_cleared_copy, continuous_columns_1, save_path=new_path)

,Feature,Skewness,Abs_skew
0,GL,0.389441,0.389441
1,WB,0.355934,0.355934
2,DR,-0.350530,0.350530
3,AM,-0.130945,0.130945
4,AI,0.063438,0.063438
5,DMP,0.020304,0.020304
6,DM,0.019432,0.019432
7,DFS,-0.000636,0.000636


In [ ]:
dataset_cleared_copy

,PAS,BT,SEX,WB,DS,RP,TNR,DR,GL,DMP,DFS,NPA,AI,DI,MI,YI,LP,TP,DM,AM,CL,CI,PS
0,2,1,1,4.770944,2,1,9,8.202830,4.356831e+22,1.152087,6.093963,0,1.930030,3,3,1425,10,2,4.352277,19.60,21,2,0
1,2,1,1,4.770944,2,1,9,8.202830,4.356831e+22,1.152087,6.093963,0,1.930425,2,4,1425,10,2,4.386036,17.20,2,2,0
2,2,1,0,4.732454,2,1,8,11.121507,5.758387e+22,1.146627,6.145662,1,1.927888,2,3,1425,10,2,3.466952,30.40,21,2,0
3,2,1,0,4.732454,2,0,7,9.958425,5.959364e+22,1.148228,6.145662,0,1.926683,0,9,1425,11,2,3.507714,36.10,1,2,1
4,2,1,0,4.770944,2,1,10,8.398384,4.675000e+22,1.132001,6.170537,0,1.920583,0,2,1425,10,2,3.993677,30.60,1,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114580,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.861694,3,4,1436,5,2,4.124313,24.80,1,2,0
114581,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.862806,2,5,1436,5,2,4.181202,23.25,21,4,0
114582,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.864987,1,7,1436,5,2,4.278707,25.10,11,2,0
114583,2,1,0,5.019941,1,0,0,0.000000,5.374568e+22,1.142748,6.067069,0,1.688830,0,6,1431,1,2,3.375573,34.80,11,4,1


In [ ]:
dataset_cleared_copy.describe()

,PAS,BT,SEX,WB,DS,RP,TNR,DR,GL,DMP,DFS,NPA,AI,DI,MI,YI,LP,TP,DM,AM,CL,CI,PS
count,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,1.145850e+05,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000,114585.000000
mean,1.960396,1.024619,0.536178,4.845323,1.236610,0.159881,0.598831,5.780065,5.122359e+22,1.139696,6.074822,0.124397,1.768216,1.549522,6.462111,1432.150788,2.532923,2.007767,3.748319,40.572984,8.353798,2.816058,0.300118
std,0.195027,0.154963,0.619748,0.315259,0.476706,0.366498,1.145566,4.315006,1.079256e+22,0.009710,0.076367,0.383508,0.066937,1.109624,3.561875,3.822121,1.666753,0.088186,0.382980,10.768079,8.018813,0.937404,0.458311
min,1.000000,1.000000,0.000000,0.000000,1.000000,0.000000,0.000000,0.000000,2.133982e+13,1.051724,4.336075,0.000000,1.573830,0.000000,1.000000,1425.000000,1.000000,1.000000,2.303115,0.100000,1.000000,1.000000,0.000000
25%,2.000000,1.000000,0.000000,4.732454,1.000000,0.000000,0.000000,0.000000,4.513401e+22,1.132792,6.030040,0.000000,1.709237,1.000000,3.000000,1429.000000,1.000000,2.000000,3.441025,33.800000,1.000000,2.000000,0.000000
50%,2.000000,1.000000,0.000000,4.845579,1.000000,0.000000,0.000000,7.930521,5.013852e+22,1.139304,6.064795,0.000000,1.767861,2.000000,6.000000,1432.000000,2.000000,2.000000,3.752494,40.800000,2.000000,2.000000,0.000000
75%,2.000000,1.000000,1.000000,4.986355,1.000000,0.000000,1.000000,8.648748,5.758387e+22,1.145930,6.120149,0.000000,1.820634,3.000000,10.000000,1436.000000,3.000000,2.000000,4.048790,47.750000,11.000000,4.000000,1.000000
max,2.000000,2.000000,4.000000,10.430036,4.000000,1.000000,10.000000,21.784516,2.968955e+23,1.171969,7.030635,5.000000,1.940086,3.000000,12.000000,1438.000000,14.000000,3.000000,4.803410,130.100000,21.000000,4.000000,1.000000


In [ ]:
continuous_columns_1 = ["WB", "DR", "GL", "DMP", "DFS", "AI", "DM", "AM"]
def plot_boxplots(df, continues_col, n_rows=2, n_cols=4):
    """
    This function plots boxplots for continuous columns and
    identifies outliers using the IQR method
    """
  sns.set(style="whitegrid", font_scale=1.2)
  plt.rcParams.update({
      "axes.edgecolor": "0.45",
      "axes.linewidth": 0.9,
      "grid.color": "0.9",
      "grid.linestyle": "--",
      "grid.linewidth":0.5
  })
  fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
  axes = axes.flatten()
  colors = sns.color_palette("tab20", len(continues_col))
  for i, col in enumerate(continues_col[:n_rows*n_cols]):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 -Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = df[col][(df[col] < lower_bound) | (df[col] > upper_bound)]
    sns.boxplot(x=df[col], ax=axes[i], color=colors[i], fliersize=4,
                linewidth=1.5, boxprops=dict(alpha=0.8, edgecolor="black"),
                whiskerprops=dict(color="black", linewidth=1.2),
                capprops=dict(color="black", linewidth=1.2),
                medianprops=dict(color="yellow", linewidth=1.5))
    axes[i].scatter(outliers, [0]*len(outliers), color="red", zorder=10, s=25, alpha=0.8)
    axes[i].set_title(col, fontsize=12, fontweight="bold", pad=8)
    axes[i].set_xlabel("")
    sns.despine(ax=axes[i], left=True)
  for j in range(len(continues_col), len(axes)):
    fig.delaxes(axes[j])
  fig.suptitle("Boxplots of Continuous Features", fontsize=15, fontweight="bold", y=0.995)
  plt.tight_layout(rect=[0, 0, 1, 0.97])
  plt.savefig("/content/drive/MyDrive/project_first/BOXPLOT.png", dpi=900, bbox_inches="tight")
  plt.close()



plot_boxplots(dataset_cleared_copy, continuous_columns_1)

In [ ]:
dataset_cleared_copy.to_excel("/content/drive/MyDrive/project_first/new_cleared_data.xlsx", index=False) # saves cleared dataset

In [ ]:
file_path = "/content/drive/MyDrive/project_first/new_cleared_data.xlsx"
dataset_cleared_new = pd.read_excel(file_path) #displays cleared data
pd.set_option("display.max_columns",None) # for showing all columns
pd.set_option("display.expand_frame_repr",False) # to avoid data clutter
dataset_cleared_new

,PAS,BT,SEX,WB,DS,RP,TNR,DR,GL,DMP,DFS,NPA,AI,DI,MI,YI,LP,TP,DM,AM,CL,CI,PS
0,2,1,1,4.770944,2,1,9,8.202830,4.356831e+22,1.152087,6.093963,0,1.930030,3,3,1425,10,2,4.352277,19.60,21,2,0
1,2,1,1,4.770944,2,1,9,8.202830,4.356831e+22,1.152087,6.093963,0,1.930425,2,4,1425,10,2,4.386036,17.20,2,2,0
2,2,1,0,4.732454,2,1,8,11.121507,5.758387e+22,1.146627,6.145662,1,1.927888,2,3,1425,10,2,3.466952,30.40,21,2,0
3,2,1,0,4.732454,2,0,7,9.958425,5.959364e+22,1.148228,6.145662,0,1.926683,0,9,1425,11,2,3.507714,36.10,1,2,1
4,2,1,0,4.770944,2,1,10,8.398384,4.675000e+22,1.132001,6.170537,0,1.920583,0,2,1425,10,2,3.993677,30.60,1,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114580,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.861694,3,4,1436,5,2,4.124313,24.80,1,2,0
114581,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.862806,2,5,1436,5,2,4.181202,23.25,21,4,0
114582,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.864987,1,7,1436,5,2,4.278707,25.10,11,2,0
114583,2,1,0,5.019941,1,0,0,0.000000,5.374568e+22,1.142748,6.067069,0,1.688830,0,6,1431,1,2,3.375573,34.80,11,4,1


In [ ]:
dataset_cleared_new_copy = dataset_cleared_new.copy() #maekes a copy of dataset

In [ ]:
def report_outlier_percentage(df, columns=None):
    """
    This function calculates the percentage of outlier 
    values in continuous columns by using IQR method
    """
  for col in columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound))
    percent = outliers.sum() / len(df) * 100
    print(f"{col}: {percent:6.2f}%")


continuous_columns_2 = ["WB", "DR", "GL", "DMP", "DFS", "AI", "DM", "AM"]
report_outlier_percentage(dataset_cleared_new_copy, continuous_columns_2)

WB:   6.63%
DR:   0.00%
GL:   3.15%
DMP:   0.39%
DFS:   1.79%
AI:   0.00%
DM:   0.00%
AM:   1.12%


In [ ]:
def remove_outliers(df, columns):
    """
    This function removes the outliers from continuous columns by using IQR
    """
  for col in columns:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
  return df



continuous_columns_3 = ["WB", "DR", "GL", "DMP", "DFS", "DM", "AM"]
dataset_cleared_new_copy = remove_outliers(dataset_cleared_new_copy, continuous_columns_3)

In [ ]:
dataset_cleared_new_copy

,PAS,BT,SEX,WB,DS,RP,TNR,DR,GL,DMP,DFS,NPA,AI,DI,MI,YI,LP,TP,DM,AM,CL,CI,PS
0,2,1,1,4.770944,2,1,9,8.202830,4.356831e+22,1.152087,6.093963,0,1.930030,3,3,1425,10,2,4.352277,19.60,21,2,0
1,2,1,1,4.770944,2,1,9,8.202830,4.356831e+22,1.152087,6.093963,0,1.930425,2,4,1425,10,2,4.386036,17.20,2,2,0
2,2,1,0,4.732454,2,1,8,11.121507,5.758387e+22,1.146627,6.145662,1,1.927888,2,3,1425,10,2,3.466952,30.40,21,2,0
3,2,1,0,4.732454,2,0,7,9.958425,5.959364e+22,1.148228,6.145662,0,1.926683,0,9,1425,11,2,3.507714,36.10,1,2,1
4,2,1,0,4.770944,2,1,10,8.398384,4.675000e+22,1.132001,6.170537,0,1.920583,0,2,1425,10,2,3.993677,30.60,1,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
114580,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.861694,3,4,1436,5,2,4.124313,24.80,1,2,0
114581,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.862806,2,5,1436,5,2,4.181202,23.25,21,4,0
114582,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.864987,1,7,1436,5,2,4.278707,25.10,11,2,0
114583,2,1,0,5.019941,1,0,0,0.000000,5.374568e+22,1.142748,6.067069,0,1.688830,0,6,1431,1,2,3.375573,34.80,11,4,1


In [ ]:
report_outlier_percentage(dataset_cleared_new_copy, continuous_columns_3)

WB:   1.44%
DR:   0.00%
GL:   0.00%
DMP:   0.27%
DFS:   0.04%
DM:   0.00%
AM:   0.14%


In [ ]:
total_number = 114994 - len(dataset_cleared_new_copy)
calculate_total_percent = (total_number / 114994) * 100
print(f"total percent outliers which removed from dataset:{calculate_total_percent}")
print("----------------------------------")
all_percent_of_outliers = 6.65 + 3.16 + 0.39 + 1.79 + 1.08
print(f"total percent of outliers calculated by IQR:{all_percent_of_outliers}")

total percent outliers which removed from dataset:11.529297180722473
----------------------------------
total percent of outliers calculated by IQR:13.070000000000002


In [ ]:
#plots the distribution diagrams for continuous columns after removing outliers
n_cols = 4
n_rows = 2
continuous_columns_after_outliers = ["WB", "DR", "GL", "DMP", "DFS", "AI", "DM", "AM"]
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols*4, n_rows*3))
axes = axes.flatten()
colors_cont_2 = sns.color_palette("tab20", len(continuous_columns_after_outliers))
for i, col in enumerate(continuous_columns_after_outliers):
  sns.histplot(dataset_cleared_new_copy[col], kde=True, bins=30, color=colors_cont_2[i], ax=axes[i])
  axes[i].set_title(col, fontsize=12, fontweight="bold")
  axes[i].set_xlabel('')
  axes[i].set_ylabel("Count")
  axes[i].grid(axis='y', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig("/content/drive/MyDrive/project_first/continuous_cols_distributions_after_outlier.png", dpi=900)
plt.close()

In [ ]:
categorical_cols_category = ["PAS", "BT", "SEX", "DS", "RP", "NPA",
                             "TP", "CI", "TNR", "MI", "YI", "LP", "DI", "CL", "PS"]

In [ ]:
cont_columns = ["WB", "DR", "GL", "DMP", "DFS", "AI", "DM", "AM"]
cat_columns = ["PAS", "BT", "SEX", "DS", "RP", "TNR", "NPA", "DI", "MI",
               "YI", "LP", "TP", "CL", "CI"]
target_col = ["PS"]
def plot_mi(df, continuous_cols, cat_cols, target, save_path=None):
    """
    This function calculates the mutual information value for each column
    then plots the diagram for it
    """
  all_features = continuous_cols + cat_cols
  X = df[all_features]
  Y = df[target]
  discrete_features = [False] * len(continuous_cols) + [True] * len(cat_cols)
  mi_values = mutual_info_classif(X, Y, discrete_features=discrete_features, random_state=42)
  mi_table = pd.DataFrame({
      "Feature": all_features,
      "Mutual Information": mi_values
  }).sort_values(by="Mutual Information", ascending=False)
  plt.figure(figsize=(12, 7))
  sns.set(style="whitegrid")
  bar_colors = sns.color_palette("viridis", len(mi_table))
  bars = plt.barh(mi_table["Feature"], mi_table["Mutual Information"], color=bar_colors)
  plt.xlabel("Mutual Information", fontsize=12)
  plt.ylabel("Features", fontsize=12)
  plt.title(f" Mutual Information Analysis of {target}", fontsize=16, fontweight="bold")
  plt.gca().invert_yaxis()
  for bar, mi_val in zip(bars, mi_table["Mutual Information"]):
    plt.text(bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
             f"{mi_val:.4f}", va="center", fontsize=10)
    plt.tight_layout()
    if save_path:
      os.makedirs(save_path, exist_ok=True)
      plt.savefig(os.path.join(save_path, "mutual_information.png"), dpi=900, bbox_inches="tight")
    plt.close()
    return mi_table


first_path = "/content/drive/MyDrive/project_first/"
mi_df = plot_mi(dataset_cleared_new_copy, cont_columns, cat_columns, target_col, save_path=first_path)
print(mi_df)

   Feature  Mutual Information
1       DR        4.588281e-03
16      MI        3.353389e-03
0       WB        3.225144e-03
5       AI        2.863882e-03
6       DM        2.743486e-03
2       GL        2.249450e-03
18      LP        1.749512e-03
13     TNR        6.720195e-04
17      YI        4.747712e-04
11      DS        3.495461e-04
21      CI        2.030649e-04
19      TP        2.017180e-04
7       AM        1.780043e-04
12      RP        1.667567e-04
4      DFS        1.288772e-04
10     SEX        1.045187e-04
14     NPA        9.153286e-05
15      DI        6.877259e-05
8      PAS        5.281151e-05
20      CL        4.164013e-06
9       BT        4.523624e-08
3      DMP        0.000000e+00


In [ ]:
def cramers_v(x, y):
    """
    This function calculates the Cramer's v value
    for caegorical columns then plots heatmap for them
    """
  correlation_matrix = pd.crosstab(x, y)
  chi2 = chi2_contingency(correlation_matrix)[0]
  n = correlation_matrix.sum().sum()
  r, k = correlation_matrix.shape
  return np.sqrt(chi2 / (n * (min(k-1, r-1))))

def plot_cramers_v(df, cate_cols, tar_col, save_path=None):
  all_cols = cate_cols + [tar_col]
  cramers_matrix = pd.DataFrame(np.zeros((len(all_cols), len(all_cols))),
                                index=all_cols, columns=all_cols)
  for col1 in all_cols:
    for col2 in all_cols:
      if col1 == col2:
        cramers_matrix.loc[col1, col2] = 1.0
      else:
        cramers_matrix.loc[col1, col2] = cramers_v(df[col1], df[col2])
  plt.figure(figsize=(12, 10))
  sns.set(style="white")
  cmap = sns.diverging_palette(220, 20, as_cmap=True)
  ax = sns.heatmap(cramers_matrix, annot=True, fmt=".2f", cmap=cmap, cbar=True,
                   linewidths=0.5, linecolor="gray", square=True)
  plt.title("Cramers V Correlation Matrix", fontsize=16, fontweight="bold")
  plt.xticks(rotation=45, ha="right")
  plt.yticks(rotation=0)
  plt.tight_layout()
  if save_path:
    os.makedirs(save_path, exist_ok=True)
    plt.savefig(os.path.join(save_path, "cramers_v_heatmap.png"), dpi=900, bbox_inches="tight")
  plt.close()
  return cramers_matrix



cat_columns = ["PAS", "BT", "SEX", "DS", "RP", "TNR", "NPA", "DI", "MI",
               "YI", "LP", "TP", "CL", "CI"]
target_col = "PS"

cramers_mat = plot_cramers_v(dataset_cleared_new_copy, cat_columns, target_col, save_path=first_path)
print(cramers_mat)

          PAS        BT       SEX        DS        RP       TNR       NPA        DI        MI        YI        LP        TP        CL        CI        PS
PAS  1.000000  0.006951  0.036335  0.209663  0.032248  0.040016  0.013740  0.005510  0.009539  0.082284  0.103976  0.008253  0.002904  0.017241  0.010125
BT   0.006951  1.000000  1.000000  0.008268  0.031148  0.038439  0.005228  0.005099  0.010772  0.018350  0.023938  0.001417  0.002728  0.006106  0.000057
SEX  0.036335  1.000000  1.000000  0.057174  0.047175  0.031396  0.008933  0.006011  0.009581  0.026505  0.034694  0.005416  0.006658  0.008559  0.014411
DS   0.209663  0.008268  0.057174  1.000000  0.395007  0.156992  0.014953  0.003303  0.031520  0.327137  0.075061  0.019996  0.002456  0.055798  0.026220
RP   0.032248  0.031148  0.047175  0.395007  1.000000  0.619520  0.031229  0.006306  0.037904  0.569719  0.055078  0.021213  0.008935  0.102069  0.018134
TNR  0.040016  0.038439  0.031396  0.156992  0.619520  1.000000  0.062680  0

In [ ]:
dataset_cleared_new_copy.to_excel("/content/drive/MyDrive/project_first/final_cleared_data.xlsx", index=False) #saves the cleared data

In [ ]:
file_path = "/content/drive/MyDrive/project_first/final_cleared_data.xlsx"
dataset_cleared_final = pd.read_excel(file_path) #displays cleared data
pd.set_option("display.max_columns",None) # for showing all columns
pd.set_option("display.expand_frame_repr",False) # to avoid data clutter
dataset_cleared_final

,PAS,BT,SEX,WB,DS,RP,TNR,DR,GL,DMP,DFS,NPA,AI,DI,MI,YI,LP,TP,DM,AM,CL,CI,PS
0,2,1,1,4.770944,2,1,9,8.202830,4.356831e+22,1.152087,6.093963,0,1.930030,3,3,1425,10,2,4.352277,19.60,21,2,0
1,2,1,1,4.770944,2,1,9,8.202830,4.356831e+22,1.152087,6.093963,0,1.930425,2,4,1425,10,2,4.386036,17.20,2,2,0
2,2,1,0,4.732454,2,1,8,11.121507,5.758387e+22,1.146627,6.145662,1,1.927888,2,3,1425,10,2,3.466952,30.40,21,2,0
3,2,1,0,4.732454,2,0,7,9.958425,5.959364e+22,1.148228,6.145662,0,1.926683,0,9,1425,11,2,3.507714,36.10,1,2,1
4,2,1,0,4.770944,2,1,10,8.398384,4.675000e+22,1.132001,6.170537,0,1.920583,0,2,1425,10,2,3.993677,30.60,1,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101731,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.861694,3,4,1436,5,2,4.124313,24.80,1,2,0
101732,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.862806,2,5,1436,5,2,4.181202,23.25,21,4,0
101733,2,1,1,4.881791,2,0,0,10.430736,5.374568e+22,1.145569,6.085079,0,1.864987,1,7,1436,5,2,4.278707,25.10,11,2,0
101734,2,1,0,5.019941,1,0,0,0.000000,5.374568e+22,1.142748,6.067069,0,1.688830,0,6,1431,1,2,3.375573,34.80,11,4,1


In [ ]:
dataset_cleared_final_copy = dataset_cleared_final.copy()

In [ ]:
dataset_cleared_final_copy.drop(["CL", "BT", "DMP"], axis=1, inplace=True) #removes the columns CL,BT, and DMP

In [ ]:
target_column11 = "PS"
high_card_col = ["TNR", "LP"]
rare_threshold = 0.01
def analyze_categorical(df, col, target_col_11, rare_threshold=rare_threshold):
    """
    This function analyzes a categorical column by diaplaying
    the distribution of its categories, identifying rare categories below
    a specified threshold, and computing the mean and standard deviation of 
    the target column for each category
    """
  freq = df[col].value_counts(normalize=True)
  print(f"distribusion of categories({len(freq)}): ")
  print(freq)
  rare_cats = freq[freq < rare_threshold].index
  print(f"rare categories({len(rare_cats)}): {list(rare_cats)}")
  target_mean = df.groupby(col)[target_col_11].mean()
  print("target mean for each category: ")
  print(target_mean)
  std_rate = target_mean.std()
  print(f"std rate of target column between categories: {std_rate:.4f}")



for col in high_card_col:
  print(f"column:{col}")
  analyze_categorical(dataset_cleared_final_copy, col, "PS")

column:TNR
distribusion of categories(11): 
TNR
0     0.687770
1     0.186443
2     0.064589
3     0.031346
4     0.013987
5     0.007755
6     0.004315
7     0.002025
8     0.001189
9     0.000462
10    0.000118
Name: proportion, dtype: float64
rare categories(6): [5, 6, 7, 8, 9, 10]
target mean for each category: 
TNR
0     0.313387
1     0.282159
2     0.275909
3     0.264346
4     0.288124
5     0.288973
6     0.250569
7     0.281553
8     0.206612
9     0.340426
10    0.416667
Name: PS, dtype: float64
std rate of target column between categories: 0.0535
column:LP
distribusion of categories(14): 
LP
1     0.345532
2     0.271074
3     0.164878
4     0.101891
5     0.056804
6     0.030736
7     0.015766
8     0.007628
9     0.003499
10    0.001484
11    0.000550
13    0.000069
12    0.000069
14    0.000020
Name: proportion, dtype: float64
rare categories(7): [8, 9, 10, 11, 13, 12, 14]
target mean for each category: 
LP
1     0.338634
2     0.289796
3     0.287409
4     0.279471
5   

In [ ]:
def plot_stacked_bar_target_rate(df, cat_col, target_col):
    """
    This function plots stacked bar chart for the TNR column
    and LP columns to visually compare the distribution of categories and their
    relation to the target variable
    """

    prop_df = (
        df.groupby(cat_col)[target_col]
        .value_counts(normalize=True)
        .rename("proportion")
        .reset_index()
        .pivot(index=cat_col, columns=target_col, values="proportion")
        .fillna(0)
    )

    prop_df = prop_df.sort_index()


    fig, ax = plt.subplots(figsize=(9, 6))
    bottom = np.zeros(len(prop_df))
    colors = ["#5DADE2", "#48C9B0"]

    for i, col in enumerate(prop_df.columns):
        bars = ax.bar(
            prop_df.index.astype(str),
            prop_df[col],
            bottom=bottom,
            color=colors[i],
            edgecolor="black",
            width=0.7,
            label=f"{target_col} = {col}",
        )


        for bar, value, base in zip(bars, prop_df[col], bottom):
            if value > 0.02:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    base + value / 2,
                    f"{value * 100:.1f}%",
                    ha="center",
                    va="center",
                    fontsize=9,
                    color="white",
                    fontweight="bold",
                )
        bottom += prop_df[col].values


    ax.set_title(f"Distribution of {target_col} within {cat_col}", fontsize=13, pad=12)
    ax.set_xlabel(cat_col, fontsize=11)
    ax.set_ylabel("Proportion", fontsize=11)
    ax.set_ylim(0, 1.05)
    ax.legend(title="Target", loc="upper right", fontsize=9)
    ax.grid(axis="y", linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig(f"/content/drive/MyDrive/project_first/stacked_plt_{cat_col}.png", dpi=600)
    plt.close()




plot_stacked_bar_target_rate(dataset_cleared_final_copy, "TNR", "PS")
plot_stacked_bar_target_rate(dataset_cleared_final_copy, "LP", "PS")

In [ ]:
def cluster_tnr(df, cat_col, target_col, method="ward", metric="euclidean", min_samples=50):
     """
    This function plots dendrogram diagram
    """
  summary = df.groupby(cat_col)[target_col].mean().reset_index()
  linked = linkage(summary[[target_col]], method=method, metric=metric)
  sns.set_theme(style="whitegrid")
  rcParams["font.family"] = "DejaVu Sans"
  rcParams["axes.titlesize"] = 16
  rcParams["axes.labelsize"] = 13
  rcParams["xtick.labelsize"] = 10
  rcParams["ytick.labelsize"] = 10
  plt.figure(figsize=(10, 6), dpi=900)
  dend = dendrogram(linked, labels=summary[cat_col].astype(str).values,
                    leaf_rotation=0, leaf_font_size=10, above_threshold_color="gray", color_threshold=None, orientation="top")
  ax = plt.gca()
  for i, line in enumerate(ax.get_lines()):
    line.set_color(sns.color_palette("tab20")[i % 10])
    line.set_linewidth(2.2)
  plt.title(f"Dendrogram of {cat_col} Based on Target Mean {target_col}", fontsize=14, fontweight="bold")
  plt.xlabel(f"Categories of {cat_col}", fontsize=12)
  plt.ylabel("Distance", fontsize=12)
  plt.grid(alpha=0.3, linestyle="--")
  legends_labels = [f"Cluster {i+1}" for i in range(len(set(dend["color_list"])))]
  color = list(set(dend["color_list"]))
  plt.legend(handles=[plt.Line2D([0], [0], color=c, lw=3) for c in color],
             labels=legends_labels, loc="upper right", fontsize=10, frameon=True)
  plt.tight_layout()
  plt.savefig("/content/drive/MyDrive/project_first/dendrogram.png", dpi=900)
  plt.close()


summary = cluster_tnr(dataset_cleared_final_copy, "TNR", "PS")

In [ ]:
def apply_manual_merge(df, cat_col, mapping_dict, new_col=None,
                       keep_unmapped_as_original=True, unmapped_value="other", target_col=None, verbose=True):
    """
    This function merges close categories for TNR column
    and makes new column for it's result
    """
  if new_col is None:
    new_col = f"{cat_col}_merged"
  mapping_str = {str(k): v for k, v in mapping_dict.items()}
  col_as_str = df[cat_col].astype(str)
  mapped = col_as_str.map(mapping_str)
  if keep_unmapped_as_original:
    result = mapped.fillna(col_as_str)
  else:
    result = mapped.fillna(unmapped_value)
  df[new_col] = result
  if verbose and target_col is not None:
    print(" == AFTER MERGE ==")
    print("Value counts (merged):")
    print(df[new_col].value_counts)
    print("\nTarget mean per merged category:")
    print(df.groupby(new_col)[target_col].mean())
    print("\nMapping applied (sample):")
    applied = pd.Series(mapping_str).reset_index()
    applied.columns = ["orig_key", "mapped_to"]
    print(applied.head(15))
  return df


mapping_tnr = {
    10:"group1", 8:"group1",
    0:"group2", 9:"group2",
    4:"group3", 5:"group3", 2:"group3", 1:"group3", 7:"group3",
    3:"group4", 6:"group4"
}
dataset_cleared_final_copy = apply_manual_merge(dataset_cleared_final_copy, cat_col="TNR",
                                                mapping_dict=mapping_tnr, new_col="TNR_MERGED",
                                                keep_unmapped_as_original=True, target_col="PS", verbose=True)

 == AFTER MERGE ==
Value counts (merged):
<bound method IndexOpsMixin.value_counts of 0         group2
1         group2
2         group1
3         group3
4         group1
           ...  
101731    group2
101732    group2
101733    group2
101734    group2
101735    group2
Name: TNR_MERGED, Length: 101736, dtype: object>

Target mean per merged category:
TNR_MERGED
group1    0.225564
group2    0.313405
group3    0.281182
group4    0.262679
Name: PS, dtype: float64

Mapping applied (sample):
   orig_key mapped_to
0        10    group1
1         8    group1
2         0    group2
3         9    group2
4         4    group3
5         5    group3
6         2    group3
7         1    group3
8         7    group3
9         3    group4
10        6    group4


In [ ]:
merge_map = {
    "group1":0,
    "group2":1,
    "group3":2,
    "group4":3
}
dataset_cleared_final_copy["TNR_MERGED"] = dataset_cleared_final_copy["TNR_MERGED"].map(merge_map)

In [ ]:
plot_stacked_bar_target_rate(dataset_cleared_final_copy, "TNR_MERGED", "PS")

In [ ]:
#plots the distribution diagram(countplot) for the TNR_MERGED column
sorted_categories = sorted(dataset_cleared_final_copy["TNR_MERGED"].unique())
plt.figure(figsize=(10, 8), dpi=900)
ax = sns.countplot(data=dataset_cleared_final_copy, x="TNR_MERGED", order=sorted_categories, palette="Set2", edgecolor="black")
for p in ax.patches:
  height = p.get_height()
  ax.annotate(f"{int(height)}", (p.get_x() + p.get_width() / 2, height), ha="center", va="center", fontsize=10)
plt.ylabel("Frequency", fontsize=12)
plt.xlabel("TNR_MERGED", fontsize=12)
plt.title("COUNT Plot of TNR_MERGED Categories", fontsize=14, fontweight="bold")
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/project_first/distribution_of_TNRMERGED.png", dpi=900)
plt.close()

In [ ]:
# performs one-hot encoding for LP column
df_lp_encoded = pd.get_dummies(dataset_cleared_final_copy["LP"], prefix="LP", drop_first=False, dtype=int)
dataset_cleared_final_copy = pd.concat([dataset_cleared_final_copy, df_lp_encoded], axis=1)
dataset_cleared_final_copy

,PAS,SEX,WB,DS,RP,TNR,DR,GL,DFS,NPA,AI,DI,MI,YI,LP,TP,DM,AM,CI,PS,TNR_MERGED,LP_1,LP_2,LP_3,LP_4,LP_5,LP_6,LP_7,LP_8,LP_9,LP_10,LP_11,LP_12,LP_13,LP_14
0,2,1,4.770944,2,1,9,8.202830,4.356831e+22,6.093963,0,1.930030,3,3,1425,10,2,4.352277,19.60,2,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
1,2,1,4.770944,2,1,9,8.202830,4.356831e+22,6.093963,0,1.930425,2,4,1425,10,2,4.386036,17.20,2,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
2,2,0,4.732454,2,1,8,11.121507,5.758387e+22,6.145662,1,1.927888,2,3,1425,10,2,3.466952,30.40,2,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,2,0,4.732454,2,0,7,9.958425,5.959364e+22,6.145662,0,1.926683,0,9,1425,11,2,3.507714,36.10,2,1,2,0,0,0,0,0,0,0,0,0,0,1,0,0,0
4,2,0,4.770944,2,1,10,8.398384,4.675000e+22,6.170537,0,1.920583,0,2,1425,10,2,3.993677,30.60,2,0,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
101731,2,1,4.881791,2,0,0,10.430736,5.374568e+22,6.085079,0,1.861694,3,4,1436,5,2,4.124313,24.80,2,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
101732,2,1,4.881791,2,0,0,10.430736,5.374568e+22,6.085079,0,1.862806,2,5,1436,5,2,4.181202,23.25,4,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
101733,2,1,4.881791,2,0,0,10.430736,5.374568e+22,6.085079,0,1.864987,1,7,1436,5,2,4.278707,25.10,2,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
101734,2,0,5.019941,1,0,0,0.000000,5.374568e+22,6.067069,0,1.688830,0,6,1431,1,2,3.375573,34.80,4,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
dataset_cleared_final_copy = dataset_cleared_final_copy.drop(columns=["LP"])

In [ ]:
cont_columns_11 = ["WB", "DR", "GL", "DFS", "AI", "DM", "AM"]

In [ ]:
#Performs scaling for continuous columns
scaler = MinMaxScaler()
dataset_cleared_final_copy[cont_columns_11] = scaler.fit_transform(dataset_cleared_final_copy[cont_columns_11])

#SMOTE TECHNIQUE
it is required due to imbalance data in target column



In [ ]:
X = dataset_cleared_final_copy.drop(columns=["PS"])

Y = dataset_cleared_final_copy["PS"]

In [ ]:
#performs the SMOTENC method for balancing the data
cat_idx = [0, 1, 3, 4, 5, 9, 11, 12, 13, 14, 17, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32]
smote_nc = SMOTENC(categorical_features=cat_idx, random_state=42)
X_resmapled, y_resampled = smote_nc.fit_resample(X, Y)

In [ ]:
df_final = pd.concat([X_resmapled, y_resampled.rename("PS")], axis=1)
df_final

,PAS,SEX,WB,DS,RP,TNR,DR,GL,DFS,NPA,AI,DI,MI,YI,TP,DM,AM,CI,TNR_MERGED,LP_1,LP_2,LP_3,LP_4,LP_5,LP_6,LP_7,LP_8,LP_9,LP_10,LP_11,LP_12,LP_13,LP_14,PS
0,2,1,0.394371,2,1,9,0.409464,0.338761,0.550996,0,0.966327,3,3,1425,2,0.768197,0.115212,2,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
1,2,1,0.394371,2,1,9,0.409464,0.338761,0.550996,0,0.967649,2,4,1425,2,0.785543,0.072007,2,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
2,2,0,0.354425,2,1,8,0.555157,0.628844,0.695246,1,0.959152,2,3,1425,2,0.313296,0.309631,2,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
3,2,0,0.354425,2,0,7,0.497099,0.670440,0.695246,0,0.955120,0,9,1425,2,0.334240,0.412241,2,2,0,0,0,0,0,0,0,0,0,0,1,0,0,0,1
4,2,0,0.394371,2,1,10,0.419226,0.404613,0.764653,0,0.934694,0,2,1425,2,0.583939,0.313231,2,0,0,0,0,0,0,0,0,0,0,1,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141891,2,0,0.401914,1,0,0,0.364731,0.551968,0.463046,0,0.496499,1,3,1433,2,0.393434,0.548435,2,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1
141892,2,0,0.743560,1,0,0,0.000000,0.667319,0.598358,1,0.335937,3,8,1428,2,0.777258,0.216113,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
141893,2,0,0.425158,1,1,1,0.428652,0.757418,0.354700,0,0.333044,1,2,1437,2,0.343159,0.667127,3,2,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
141894,2,0,0.657286,1,1,2,0.638262,0.549404,0.596274,0,0.534491,1,2,1428,2,0.488965,0.586820,2,2,0,0,1,0,0,0,0,0,0,0,0,0,0,0,1


In [ ]:
# shuffles data
df_final = df_final.sample(frac=1, random_state=42).reset_index(drop=True)
df_final

,PAS,SEX,WB,DS,RP,TNR,DR,GL,DFS,NPA,AI,DI,MI,YI,TP,DM,AM,CI,TNR_MERGED,LP_1,LP_2,LP_3,LP_4,LP_5,LP_6,LP_7,LP_8,LP_9,LP_10,LP_11,LP_12,LP_13,LP_14,PS
0,2,0,0.720676,2,0,0,0.000000,1.000000,0.963023,0,0.190558,3,2,1431,2,0.382554,0.642664,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,2,0,0.582434,1,0,0,0.381536,0.713334,0.538629,0,0.483261,3,1,1432,2,0.245275,0.514851,1,1,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0
2,2,0,0.849743,1,0,0,0.440743,0.628844,0.695246,0,0.364178,2,12,1430,2,0.491993,0.545455,3,1,0,1,0,0,0,0,0,0,0,0,0,0,0,0,1
3,2,1,0.313618,1,0,0,0.000000,0.338761,0.326197,0,0.125261,1,7,1436,2,0.443925,0.473447,4,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
4,2,0,0.332038,1,0,0,0.000000,0.439129,0.322908,0,0.104692,0,11,1437,2,0.417677,0.533222,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
141891,2,0,0.451533,1,0,0,0.000000,0.385988,0.337820,0,0.093550,1,3,1436,2,0.370694,0.658013,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
141892,2,0,0.342431,1,0,0,0.000000,0.474746,0.331358,0,0.107254,0,5,1433,2,0.420290,0.406831,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
141893,2,0,0.415364,1,0,0,0.000000,0.648217,0.382927,0,0.139037,3,6,1434,2,0.464215,0.575014,2,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1
141894,2,1,0.483998,1,0,0,0.352977,0.524636,0.513730,1,0.669205,3,12,1430,2,0.262806,0.635943,3,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0,1


In [ ]:
#prepares settings for pycaret
target_col_pycaret = "PS"
numeric_features = ["WB", "DR", "GL", "DFS", "AI", "DM", "AM"]
categorical_features = ["PAS", "SEX", "DS", "RP", "TNR", "NPA", "DI", "MI", "YI", "TP", "CI", "TNR_MERGED",
                        "LP_1", "LP_2", "LP_3", "LP_4", "LP_5", "LP_6",
                        "LP_7", "LP_8", "LP_9", "LP_10", "LP_11", "LP_13", "LP_14"]
s = setup(data=df_final, target=target_col_pycaret, numeric_features=numeric_features, categorical_features=categorical_features,
          preprocess=False, normalize=False, polynomial_features=False, remove_outliers=False, fold_shuffle=True,
          session_id=42, verbose=True, train_size=0.8)

,Description,Value
0,Session id,42
1,Target,PS
2,Target type,Binary
3,Original data shape,"(141896, 34)"
4,Transformed data shape,"(141896, 34)"
5,Transformed train set shape,"(113516, 34)"
6,Transformed test set shape,"(28380, 34)"
7,Numeric features,7
8,Categorical features,25


In [ ]:
best_models = compare_models(n_select=8, sort="Accuracy") # pycaret trains and test models then chooses best 8 models

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
lightgbm,Light Gradient Boosting Machine,0.7712,0.8204,0.5656,0.9606,0.7120,0.5424,0.5950,83.3480
rf,Random Forest Classifier,0.7577,0.8374,0.7282,0.7739,0.7503,0.5154,0.5163,5.1890
xgboost,Extreme Gradient Boosting,0.7522,0.8130,0.5822,0.8823,0.7015,0.5045,0.5365,0.5540
et,Extra Trees Classifier,0.7501,0.8320,0.7459,0.7523,0.7490,0.5002,0.5003,4.2810
gbc,Gradient Boosting Classifier,0.7367,0.7943,0.5911,0.8339,0.6918,0.4733,0.4948,6.4510
dt,Decision Tree Classifier,0.6605,0.6605,0.6617,0.6601,0.6609,0.3210,0.3210,0.5710
knn,K Neighbors Classifier,0.6330,0.6800,0.7121,0.6148,0.6599,0.2659,0.2693,2.9950
ada,Ada Boost Classifier,0.6215,0.6730,0.6322,0.6192,0.6255,0.2430,0.2432,1.4100
ridge,Ridge Classifier,0.5749,0.6037,0.6127,0.5696,0.5904,0.1497,0.1502,0.0370
lda,Linear Discriminant Analysis,0.5749,0.6037,0.6126,0.5696,0.5903,0.1497,0.1502,0.0950


Processing:   0%|          | 0/72 [00:00<?, ?it/s]

In [ ]:
results_df = pull() # saves pycaret's results
results_df.to_excel("/content/drive/MyDrive/project_first/pycaret_models_results.xlsx", index=False)

In [ ]:
# saves parameters which pycaret used for each models
params_list = []
for model in best_models:
  model_name = str(model).split("(")[0]
  params = model.get_params()
  params_list.append({
      "model":model_name,
      "parameters":params
  })
params_df = pd.DataFrame(params_list)
params_df.to_excel("/content/drive/MyDrive/project_first/pycaret_models_params.xlsx", index=False)

In [ ]:
#plots confusin matrix for pycaret results
x_test = get_config("X_test")
y_test = get_config("y_test")
fig, ax = plt.subplots(2, 4, figsize=(20, 10), dpi=900)
ax = ax.flatten()
for i, model in enumerate(best_models):
  preds = predict_model(model, data=x_test)
  y_pred = preds["prediction_label"]
  cm = confusion_matrix(y_test, y_pred)
  labels = np.unique(y_test)
  sns.heatmap(cm, annot=True, fmt="d", cmap="YlGnBu", cbar=False, xticklabels=labels, yticklabels=labels, ax=ax[i])
  ax[i].set_title(str(model).split('(')[0], fontsize=10)
  ax[i].set_xlabel("Predicted", fontsize=8)
  ax[i].set_ylabel("Actual", fontsize=8)
  ax[i].tick_params(axis="both", labelsize=8)
for j in range(len(best_models), len(ax)):
  fig.delaxes(ax[j])
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/project_first/confusion_matrix.png", dpi=900)
plt.close()

In [ ]:
my_df_results = pd.read_excel("/content/drive/MyDrive/project_first/pycaret_models_results.xlsx")
my_df_results

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
0,Light Gradient Boosting Machine,0.7712,0.8204,0.5656,0.9606,0.7120,0.5424,0.5950,83.348
1,Random Forest Classifier,0.7577,0.8374,0.7282,0.7739,0.7503,0.5154,0.5163,5.189
2,Extreme Gradient Boosting,0.7522,0.8130,0.5822,0.8823,0.7015,0.5045,0.5365,0.554
3,Extra Trees Classifier,0.7501,0.8320,0.7459,0.7523,0.7490,0.5002,0.5003,4.281
4,Gradient Boosting Classifier,0.7367,0.7943,0.5911,0.8339,0.6918,0.4733,0.4948,6.451
5,Decision Tree Classifier,0.6605,0.6605,0.6617,0.6601,0.6609,0.3210,0.3210,0.571
6,K Neighbors Classifier,0.6330,0.6800,0.7121,0.6148,0.6599,0.2659,0.2693,2.995
7,Ada Boost Classifier,0.6215,0.6730,0.6322,0.6192,0.6255,0.2430,0.2432,1.410
8,Ridge Classifier,0.5749,0.6037,0.6127,0.5696,0.5904,0.1497,0.1502,0.037
9,Linear Discriminant Analysis,0.5749,0.6037,0.6126,0.5696,0.5903,0.1497,0.1502,0.095


#PYCARET TOP MODELS:
1-Light Gradient Boosting Machine

2-Random Forest Classifier

3-Extreme Gradient Boosting

4-Extra Trees Classifier

5-Gradient Boosting Classifier

6-Decision Tree Classifier

7-K Neighbors Classifier

8-Ada Boost Classifier

In [ ]:
my_df_params = pd.read_excel("/content/drive/MyDrive/project_first/pycaret_models_params.xlsx")
my_df_params

,model,parameters
0,LGBMClassifier,"{'boosting_type': 'gbdt', 'class_weight': None..."
1,RandomForestClassifier,"{'bootstrap': True, 'ccp_alpha': 0.0, 'class_w..."
2,XGBClassifier,"{'objective': 'binary:logistic', 'base_score':..."
3,ExtraTreesClassifier,"{'bootstrap': False, 'ccp_alpha': 0.0, 'class_..."
4,GradientBoostingClassifier,"{'ccp_alpha': 0.0, 'criterion': 'friedman_mse'..."
5,DecisionTreeClassifier,"{'ccp_alpha': 0.0, 'class_weight': None, 'crit..."
6,KNeighborsClassifier,"{'algorithm': 'auto', 'leaf_size': 30, 'metric..."
7,AdaBoostClassifier,"{'algorithm': 'SAMME.R', 'estimator': None, 'l..."


In [ ]:
# sets all parameters pycaret used for each model again
light_gbm = LGBMClassifier(boosting_type= 'gbdt', class_weight=None,
                           colsample_bytree=1.0, importance_type='split',
                           learning_rate=0.1, max_depth= -1, min_child_samples=20,
                           min_child_weight=0.001, min_split_gain=0.0, n_estimators=100,
                           n_jobs=-1, num_leaves=31, objective=None, random_state=42,
                           reg_alpha=0.0, reg_lambda=0.0,
                           subsample=1.0, subsample_for_bin=200000, subsample_freq=0)

rf = RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None, criterion='gini', max_depth=None,
                            max_features='sqrt', max_leaf_nodes=None, max_samples=None, min_impurity_decrease=0.0,
                            min_samples_leaf=1, min_samples_split=2, min_weight_fraction_leaf=0.0,
                            monotonic_cst=None, n_estimators=100, n_jobs=-1, oob_score=False, random_state=42,
                            verbose=0, warm_start=False)

xgb = XGBRFClassifier(objective='binary:logistic', base_score=None, booster='gbtree',  device='cpu', tree_method='auto', verbosity=0)
extr = ExtraTreeClassifier(ccp_alpha=0.0, criterion='gini', max_features='sqrt',
                           min_samples_leaf=1, min_samples_split=2, min_weight_fraction_leaf=0.0,
                           random_state=42)
gb = GradientBoostingClassifier(ccp_alpha=0.0, criterion='friedman_mse', learning_rate=0.1, loss='log_loss', max_depth=3,
                                min_impurity_decrease=0.0, min_samples_leaf=1, min_samples_split=2,
                                min_weight_fraction_leaf=0.0, n_estimators=100, random_state=42, subsample=1.0,
                                tol=0.0001, validation_fraction=0.1, verbose=0, warm_start=False)
dt = DecisionTreeClassifier(ccp_alpha=0.0, class_weight=None, criterion='gini', min_impurity_decrease=0.0,
                            min_samples_leaf=1, min_samples_split=2, min_weight_fraction_leaf=0.0, random_state=42, splitter='best')
knn = KNeighborsClassifier(algorithm='auto', leaf_size=30, metric='minkowski',
                           metric_params=None, n_jobs=-1, n_neighbors=5, p=2, weights='uniform')
ada = AdaBoostClassifier(algorithm='SAMME.R', estimator=None, learning_rate=1.0, n_estimators=50, random_state=42)


In [ ]:
# uses same models and parameters for training and testing hybrid models
#first 4 best models sent to voting classifier
# the result of voting classifer and reset of 4 models sent to stacking classifier
hybrid1 = VotingClassifier(estimators=[("LIGHTGBM", light_gbm), ("RF", rf), ("EXGBOOST",xgb), ("EXTR", extr)], voting="soft")
hybrid2 = StackingClassifier(estimators=[("voting", hybrid1), ("GB", gb), ("DT", dt), ("KNN", knn), ("ADA", ada)],
                             final_estimator=gb, cv=10)
start_time = time.time()
hybrid2.fit(get_config("X_train"), get_config("y_train"))
end_time = time.time()
training_time = end_time - start_time

[LightGBM] [Info] Number of positive: 56758, number of negative: 56758
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.019103 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1881
[LightGBM] [Info] Number of data points in the train set: 113516, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Number of positive: 51082, number of negative: 51082
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.006670 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1881
[LightGBM] [Info] Number of data points in the train set: 102164, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0

In [ ]:
# saves all results
preds = hybrid2.predict(get_config("X_test"))
probs = hybrid2.predict_proba(get_config("X_test"))[:, 1]
y_true = get_config("y_test")
F1 = f1_score(y_true, preds, average="binary")
Accuracy = accuracy_score(y_true, preds)
AUC = roc_auc_score(y_true, probs)
Recall = recall_score(y_true, preds, average="binary")
Precision = precision_score(y_true, preds, average="binary")
MCC = matthews_corrcoef(y_true, preds)
Kappa = cohen_kappa_score(y_true, preds)
metrics = {
    "Model":"STACKING_VOTING_HYBRID ",
    "Accuracy":Accuracy,
    "AUC":AUC,
    "Recall":Recall,
    "Prec.":Precision,
    "F1":F1,
    "Kappa":Kappa,
    "MCC":MCC,
    "TT (Sec)":training_time
}
metrics_data = pd.DataFrame([metrics])
df_all_models = pd.concat([my_df_results, metrics_data], ignore_index=True)
df_all_models.to_excel("/content/drive/MyDrive/project_first/df_all_models.xlsx", index=False)

In [ ]:
df_all = pd.read_excel("/content/drive/MyDrive/project_first/df_all_models.xlsx")
df_all

,Model,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC,TT (Sec)
0,Light Gradient Boosting Machine,0.771200,0.820400,0.565600,0.960600,0.712000,0.542400,0.595000,83.348000
1,Random Forest Classifier,0.757700,0.837400,0.728200,0.773900,0.750300,0.515400,0.516300,5.189000
2,Extreme Gradient Boosting,0.752200,0.813000,0.582200,0.882300,0.701500,0.504500,0.536500,0.554000
3,Extra Trees Classifier,0.750100,0.832000,0.745900,0.752300,0.749000,0.500200,0.500300,4.281000
4,Gradient Boosting Classifier,0.736700,0.794300,0.591100,0.833900,0.691800,0.473300,0.494800,6.451000
5,Decision Tree Classifier,0.660500,0.660500,0.661700,0.660100,0.660900,0.321000,0.321000,0.571000
6,K Neighbors Classifier,0.633000,0.680000,0.712100,0.614800,0.659900,0.265900,0.269300,2.995000
7,Ada Boost Classifier,0.621500,0.673000,0.632200,0.619200,0.625500,0.243000,0.243200,1.410000
8,Ridge Classifier,0.574900,0.603700,0.612700,0.569600,0.590400,0.149700,0.150200,0.037000
9,Linear Discriminant Analysis,0.574900,0.603700,0.612600,0.569600,0.590300,0.149700,0.150200,0.095000


In [ ]:
# plots confusion matrix for hybrid model
cm = confusion_matrix(y_true, preds)
labels_h = np.unique(y_true)
plt.figure(figsize=(6, 5), dpi=900)
sns.heatmap(cm, annot=True, fmt='d', cmap="Blues", cbar=False, xticklabels=labels_h, yticklabels=labels_h)
plt.title("Confusion Matrix - Hybrid Stacking", fontsize=14)
plt.xlabel("Predicted", fontsize=12)
plt.ylabel("Actual", fontsize=12)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/project_first/Hybrid_confusion_matrix.png", dpi=900)
plt.close()

In [ ]:
#plots radar diagram
top8_names = df_all["Model"].tolist()[:8]
hybrid_name = df_all["Model"].iloc[-1]
models_plot = top8_names + [hybrid_name]
df_plot = df_all[df_all["Model"].isin(models_plot)].reset_index(drop=True)
metrics = ["Recall", "Prec.", "F1", "Kappa", "MCC"]
N = len(metrics)
angles = [n / float(N) * 2 * pi for n in range(N)]
angles += angles[:1]
plt.figure(figsize=(10, 10), dpi=900)
ax = plt.subplot(111, polar=True)
for i, row in df_plot.iterrows():
  values = row[metrics].tolist()
  values += values[:1]
  ax.plot(angles, values, linewidth=2, label=row["Model"])
  ax.fill(angles, values, alpha=0.1)
ax.set_theta_offset(pi / 2)
ax.set_theta_direction(-1)
plt.xticks(angles[:-1], metrics, fontsize=12)
ax.set_rlabel_position(0)
plt.yticks([0.2, 0.4, 0.6, 0.8, 1.0], ["0.2", "0.4", "0.6", "0.8", "1.0"], color="gray", size=10)
plt.ylim(0,1)
plt.title("Comparison of Models- Radar chart", fontsize=16)
plt.legend(loc="upper right", bbox_to_anchor=(1.3, 1.1))
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/project_first/Radar_chart.png", dpi=900)
plt.close()

In [ ]:
# plots barchart
df_plot_sorted = df_plot.sort_values("Accuracy", ascending=False).reset_index(drop=True)
plt.figure(figsize=(12, 6), dpi=900)
sns.set_style("whitegrid")
palette = sns.color_palette("tab10", n_colors=len(df_plot_sorted))
ax = sns.barplot(x="Model", y="Accuracy", data=df_plot_sorted, palette=palette)
for p in ax.patches:
  height = p.get_height()
  ax.text(
      x=p.get_x()+p.get_width()/2,
      y=height+0.005,
      s=f"{height:.3f}",
      ha="center",
      fontsize=10,
      weight="bold"
  )
plt.xticks(rotation=45, ha="right", fontsize=12)
plt.ylabel("Accuracy", fontsize=12, weight="bold")
plt.xlabel("Model", fontsize=12, weight="bold")
plt.title("Comparison of Model Accuracy", fontsize=16, weight="bold")
plt.grid(axis='y', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/project_first/bar_plots_1.png", dpi=900)
plt.close()

In [ ]:
# prints the system's information used for the projects
metrics_1 = ["Accuracy", "AUC", "Recall", "Prec.", "F1", "Kappa", "MCC", "TT (Sec)"]
plt.figure(figsize=(14, len(df_all)*0.5+4), dpi=900)
sns.set_style("whitegrid")
cmap = sns.color_palette("coolwarm", as_cmap=True)
ax = sns.heatmap(
    df_all.set_index("Model")[metrics_1],
    annot=True,
    fmt=".3f",
    cmap=cmap,
    cbar=True,
    linewidths=0.5,
    linecolor="gray"
)
plt.title("Model Metrics Heatmap", fontsize=18, weight="bold")
plt.xticks(fontsize=12, rotation=45, ha="right")
plt.yticks(fontsize=12)
plt.tight_layout()
plt.savefig("/content/drive/MyDrive/project_first/heatmap.png", dpi=900)
plt.close()

In [ ]:
# prints the system's information used for the projects
def run_cmd(cmd):
    try:
        out = subprocess.check_output(cmd, shell=True, text=True)
        return out.strip()
    except Exception:
        return "N/A"


def get_full_system_info():
    info = {}


    info["Operating System"] = platform.platform()
    info["Python Version"] = platform.python_version()
    info["Architecture"] = platform.machine()


    cpu_info = get_cpu_info()
    info["CPU Brand"] = cpu_info.get("brand_raw", "N/A")
    info["CPU Architecture"] = cpu_info.get("arch", "N/A")
    info["CPU Cores (Logical)"] = psutil.cpu_count(logical=True)
    info["CPU Cores (Physical)"] = psutil.cpu_count(logical=False)


    freq = psutil.cpu_freq()
    if freq:
        info["CPU Frequency (Current MHz)"] = round(freq.current, 2)
        info["CPU Frequency (Max MHz)"] = round(freq.max, 2)
    else:
        info["CPU Frequency (Current MHz)"] = "N/A"


    info["CPU Usage (%)"] = psutil.cpu_percent(interval=1)


    mem = psutil.virtual_memory()
    info["RAM Total (GB)"] = round(mem.total / (1024**3), 2)
    info["RAM Available (GB)"] = round(mem.available / (1024**3), 2)
    info["RAM Used (GB)"] = round(mem.used / (1024**3), 2)
    info["RAM Usage (%)"] = mem.percent


    dmidecode = run_cmd("sudo dmidecode --type memory | grep 'Type:' | head -n 1")
    info["RAM Type"] = dmidecode.replace("Type:", "").strip() if "Error" not in dmidecode else "N/A"


    try:
        disk = psutil.disk_usage('/')
        info["Storage Total (GB)"] = round(disk.total / (1024**3), 2)
        info["Storage Used (GB)"] = round(disk.used / (1024**3), 2)
        info["Storage Free (GB)"] = round(disk.free / (1024**3), 2)
        info["Storage Usage (%)"] = disk.percent
    except Exception:
        info["Storage Total (GB)"] = "N/A"


    storage_type = run_cmd("cat /sys/block/sda/queue/rotational")  # 0=SSD, 1=HDD
    if storage_type == "0":
        info["Storage Type"] = "SSD / NVMe"
    elif storage_type == "1":
        info["Storage Type"] = "HDD"
    else:
        info["Storage Type"] = "Unknown"


    info["CPU Detailed Info (lscpu)"] = run_cmd("lscpu")

    return info


system_info = get_full_system_info()


df = pd.DataFrame(list(system_info.items()), columns=["Property", "Value"])
display(df)


print("\n========== CPU ==========\n")
print(system_info["CPU Detailed Info (lscpu)"])

,Property,Value
0,Operating System,Linux-6.6.105+-x86_64-with-glibc2.35
1,Python Version,3.12.12
2,Architecture,x86_64
3,CPU Brand,Intel(R) Xeon(R) CPU @ 2.20GHz
4,CPU Architecture,X86_64
5,CPU Cores (Logical),8
6,CPU Cores (Physical),4
7,CPU Frequency (Current MHz),2200.16
8,CPU Frequency (Max MHz),0.0
9,CPU Usage (%),0.8



========== CPU ==========

Architecture:                            x86_64
CPU op-mode(s):                          32-bit, 64-bit
Address sizes:                           46 bits physical, 48 bits virtual
Byte Order:                              Little Endian
CPU(s):                                  8
On-line CPU(s) list:                     0-7
Vendor ID:                               GenuineIntel
Model name:                              Intel(R) Xeon(R) CPU @ 2.20GHz
CPU family:                              6
Model:                                   79
Thread(s) per core:                      2
Core(s) per socket:                      4
Socket(s):                               1
Stepping:                                0
BogoMIPS:                                4400.32
Flags:                                   fpu vme de pse tsc msr pae mce cx8 apic sep mtrr pge mca cmov pat pse36 clflush mmx fxsr sse sse2 ss ht syscall nx pdpe1gb rdtscp lm constant_tsc rep_good nopl xtopology nonst

In [ ]:
#saves and prints the libraries used in this project
packages = sorted([(d.project_name, d.version) for d in pkg_resources.working_set])
df_libs = pd.DataFrame(packages, columns=["Library", "Version"])
print("=== Library Versions ===")
display(df_libs.head(20))
df_libs.to_excel("/content/drive/MyDrive/project_first/library_versions.xlsx", index=False)

=== Library Versions ===


,Library,Version
0,Bottleneck,1.4.2
1,Brotli,1.1.0
2,Cython,3.0.12
3,Farama-Notifications,0.0.4
4,GDAL,3.8.4
5,PyDrive2,1.21.3
6,PyGObject,3.42.0
7,PyJWT,2.10.1
8,PySocks,1.7.1
9,Send2Trash,1.8.3
